In [10]:
pip install pymupdf


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 98.6 MB/s eta 0:00:00:00:01

[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [12]:
import os
import re
import hashlib
import time
from typing import List, Dict, Tuple
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
import PyPDF2
import fitz  # PyMuPDF - more reliable for complex PDFs
from pathlib import Path

# ---------- CONFIG ----------
PDF_PATH = "IBMTurbonomic_API_ep.pdf"  # Local path to save/load PDF
CHROMA_PATH = "/data/wikidata/chromadb"
COLLECTION_NAME = "turbo_api"
EMBEDDING_FN = SentenceTransformerEmbeddingFunction("all-MiniLM-L6-v2")

# Chunking parameters
MIN_CHUNK_SIZE = 500  # Minimum characters per chunk
MAX_CHUNK_SIZE = 2000  # Maximum characters per chunk
OVERLAP_SIZE = 100  # Characters to overlap between chunks
DEBUG = False  # Set to True to print chunks instead of storing

class TurbonomicPDFProcessor:
    def __init__(self, pdf_path: str, chroma_path: str, collection_name: str):
        self.pdf_path = pdf_path
        self.chroma_path = chroma_path
        self.collection_name = collection_name
        self.embedding_fn = EMBEDDING_FN
        
        # Initialize ChromaDB
        if not DEBUG:
            self.client = chromadb.PersistentClient(path=chroma_path)
            self.collection = self.client.get_or_create_collection(
                name=collection_name,
                embedding_function=self.embedding_fn
            )
        
        # Track processing stats
        self.stats = {
            'total_pages': 0,
            'total_chunks': 0,
            'processed_pages': 0,
            'skipped_pages': 0,
            'errors': []
        }
    
    def download_pdf(self, url: str) -> bool:
        """Download PDF from Box URL if not already present."""
        if os.path.exists(self.pdf_path):
            print(f"✅ PDF already exists at {self.pdf_path}")
            return True
        
        try:
            import requests
            print(f"📥 Downloading PDF from {url}...")
            
            # Note: Box URLs might require special handling
            # You might need to get the direct download link
            response = requests.get(url, stream=True)
            response.raise_for_status()
            
            with open(self.pdf_path, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    f.write(chunk)
            
            print(f"✅ PDF downloaded successfully to {self.pdf_path}")
            return True
            
        except Exception as e:
            print(f"❌ Error downloading PDF: {e}")
            print("📝 Please manually download the PDF and place it in the same directory as this script")
            return False
    
    def extract_text_from_pdf(self) -> List[Dict]:
        """Extract text from PDF using PyMuPDF (more reliable for complex PDFs)."""
        pages_content = []
        
        try:
            doc = fitz.open(self.pdf_path)
            self.stats['total_pages'] = len(doc)
            
            print(f"📄 Processing {len(doc)} pages...")
            
            for page_num in range(len(doc)):
                try:
                    page = doc.load_page(page_num)
                    text = page.get_text()
                    
                    # Clean and normalize text
                    cleaned_text = self.clean_text(text)
                    
                    if len(cleaned_text.strip()) > 50:  # Only process pages with substantial content
                        pages_content.append({
                            'page_number': page_num + 1,
                            'text': cleaned_text,
                            'raw_text': text
                        })
                        self.stats['processed_pages'] += 1
                    else:
                        self.stats['skipped_pages'] += 1
                    
                    # Progress indicator
                    if (page_num + 1) % 100 == 0:
                        print(f"📖 Processed {page_num + 1}/{len(doc)} pages...")
                
                except Exception as e:
                    error_msg = f"Error processing page {page_num + 1}: {str(e)}"
                    print(f"⚠️ {error_msg}")
                    self.stats['errors'].append(error_msg)
                    continue
            
            doc.close()
            print(f"✅ Successfully extracted text from {self.stats['processed_pages']} pages")
            return pages_content
            
        except Exception as e:
            print(f"❌ Error opening PDF: {e}")
            return []
    
    def clean_text(self, text: str) -> str:
        """Clean and normalize extracted text."""
        # Remove excessive whitespace
        text = re.sub(r'\s+', ' ', text)
        
        # Remove special characters that might interfere
        text = re.sub(r'[^\w\s\-.,;:()[\]{}/"\'=<>@#$%&*+?!]', ' ', text)
        
        # Normalize line breaks
        text = re.sub(r'\n+', '\n', text)
        
        # Remove leading/trailing whitespace
        text = text.strip()
        
        return text
    
    def detect_section_boundaries(self, text: str) -> List[str]:
        """Detect logical section boundaries in API documentation."""
        # Common API documentation section markers
        section_patterns = [
            r'\n\s*(?:CHAPTER|Chapter)\s+\d+',
            r'\n\s*(?:SECTION|Section)\s+\d+',
            r'\n\s*(?:API|Api)\s+(?:Reference|Endpoint|Method)',
            r'\n\s*(?:GET|POST|PUT|DELETE|PATCH)\s+/',
            r'\n\s*(?:Request|Response)\s+(?:Parameters|Body|Headers)',
            r'\n\s*(?:Example|Examples)\s*:',
            r'\n\s*(?:Description|Overview)\s*:',
            r'\n\s*(?:Authentication|Authorization)',
            r'\n\s*(?:Error|Errors)\s+(?:Codes|Messages)',
            r'\n\s*[A-Z][A-Z\s]{10,}',  # All caps headers
            r'\n\s*\d+\.\d+\s+[A-Z]',   # Numbered sections
        ]
        
        # Find all potential split points
        split_points = [0]
        
        for pattern in section_patterns:
            matches = re.finditer(pattern, text, re.IGNORECASE)
            for match in matches:
                split_points.append(match.start())
        
        # Sort and deduplicate
        split_points = sorted(list(set(split_points)))
        split_points.append(len(text))
        
        # Create sections
        sections = []
        for i in range(len(split_points) - 1):
            start = split_points[i]
            end = split_points[i + 1]
            section_text = text[start:end].strip()
            
            if len(section_text) > MIN_CHUNK_SIZE:
                sections.append(section_text)
        
        return sections if sections else [text]  # Fallback to whole text if no sections found
    
    def create_chunks(self, pages_content: List[Dict]) -> List[Dict]:
        """Chunk each page using simple character-based sliding window."""
        chunks = []

        for page_data in pages_content:
            page_text = page_data['text']
            page_num = page_data['page_number']
            text = page_text.strip()

            start = 0
            chunk_idx = 0

            while start < len(text):
                end = start + MAX_CHUNK_SIZE
                chunk_text = text[start:end].strip()

                if len(chunk_text) >= MIN_CHUNK_SIZE:
                    chunks.append({
                        'chunk_id': f"page_{page_num}_chunk_{chunk_idx}",
                        'content': chunk_text,
                        'metadata': {
                            'page_number': page_num,
                            'chunk_index': chunk_idx,
                            'chunk_type': 'sliding_window',
                            'source': 'turbonomic_api_docs',
                            'char_count': len(chunk_text)
                        }
                    })
                    chunk_idx += 1

                # Slide window forward with overlap
                start = end - OVERLAP_SIZE

        self.stats['total_chunks'] = len(chunks)
        return chunks

    
    def split_large_section(self, text: str) -> List[str]:
        """Split large sections into smaller chunks with overlap."""
        if len(text) <= MAX_CHUNK_SIZE:
            return [text]
        
        chunks = []
        start = 0
        
        while start < len(text):
            # Find a good breaking point (sentence end, paragraph, etc.)
            end = start + MAX_CHUNK_SIZE
            
            if end >= len(text):
                # Last chunk
                chunks.append(text[start:].strip())
                break
            
            # Try to find a good breaking point
            break_point = end
            
            # Look for sentence endings
            for i in range(end, max(start + MIN_CHUNK_SIZE, end - 200), -1):
                if text[i:i+2] in ['. ', '.\n', '? ', '?\n', '! ', '!\n']:
                    break_point = i + 1
                    break
            
            # If no sentence ending found, look for paragraph breaks
            if break_point == end:
                for i in range(end, max(start + MIN_CHUNK_SIZE, end - 200), -1):
                    if text[i:i+2] == '\n\n':
                        break_point = i + 2
                        break
            
            chunks.append(text[start:break_point].strip())
            start = break_point - OVERLAP_SIZE if break_point > start + OVERLAP_SIZE else break_point
        
        return [chunk for chunk in chunks if len(chunk.strip()) > MIN_CHUNK_SIZE]
    
    def store_chunks_in_chromadb(self, chunks: List[Dict]) -> None:
        """Store chunks in ChromaDB."""
        if DEBUG:
            print("🐛 DEBUG MODE: Showing first 3 chunks...")
            for i, chunk in enumerate(chunks[:3]):
                print(f"\n--- Chunk {i+1} ---")
                print(f"ID: {chunk['chunk_id']}")
                print(f"Metadata: {chunk['metadata']}")
                print(f"Content (first 200 chars): {chunk['content'][:200]}...")
                print("-" * 50)
            return
        
        print(f"💾 Storing {len(chunks)} chunks in ChromaDB...")
        
        batch_size = 100
        for i in range(0, len(chunks), batch_size):
            batch = chunks[i:i + batch_size]
            
            try:
                # Prepare batch data
                documents = [chunk['content'] for chunk in batch]
                ids = [chunk['chunk_id'] for chunk in batch]
                metadatas = [chunk['metadata'] for chunk in batch]
                
                # Store in ChromaDB
                self.collection.upsert(
                    documents=documents,
                    ids=ids,
                    metadatas=metadatas
                )
                
                print(f"✅ Stored batch {i//batch_size + 1}/{(len(chunks) + batch_size - 1)//batch_size}")
                
            except Exception as e:
                error_msg = f"Error storing batch {i//batch_size + 1}: {str(e)}"
                print(f"❌ {error_msg}")
                self.stats['errors'].append(error_msg)
                continue
    
    def process_pdf(self) -> Dict:
        """Main processing function."""
        print("🚀 Starting Turbonomic PDF processing...")
        
        # Download PDF if needed
        if not os.path.exists(self.pdf_path):
            return {'success': False, 'error': 'PDF doesnt exist'}
        
        # Extract text from PDF
        pages_content = self.extract_text_from_pdf()
        if not pages_content:
            return {'success': False, 'error': 'Failed to extract text from PDF'}
        
        # Create chunks
        print("🔄 Creating logical chunks...")
        chunks = self.create_chunks(pages_content)
        
        if not chunks:
            return {'success': False, 'error': 'No chunks created'}
        
        # Store in ChromaDB
        self.store_chunks_in_chromadb(chunks)
        
        # Print statistics
        print(f"\n📊 Processing Statistics:")
        print(f"   📄 Total pages: {self.stats['total_pages']}")
        print(f"   ✅ Processed pages: {self.stats['processed_pages']}")
        print(f"   ⏭️  Skipped pages: {self.stats['skipped_pages']}")
        print(f"   📦 Total chunks: {self.stats['total_chunks']}")
        print(f"   ❌ Errors: {len(self.stats['errors'])}")
        
        if self.stats['errors']:
            print(f"\n⚠️ Errors encountered:")
            for error in self.stats['errors'][:5]:  # Show first 5 errors
                print(f"   - {error}")
            if len(self.stats['errors']) > 5:
                print(f"   ... and {len(self.stats['errors']) - 5} more errors")
        
        return {
            'success': True,
            'stats': self.stats,
            'chunks_created': len(chunks)
        }
    
    def test_chromadb_connection(self) -> None:
        """Test ChromaDB connection and show collection info."""
        if DEBUG:
            print("🐛 DEBUG mode - skipping ChromaDB test")
            return
        
        try:
            collections = self.client.list_collections()
            print(f"\n🗂️  Collections in ChromaDB:")
            for col in collections:
                print(f"   - {col.name}")
            
            # Check our collection
            if self.collection_name in [col.name for col in collections]:
                collection = self.client.get_collection(self.collection_name)
                count = collection.count()
                print(f"\n📦 Collection '{self.collection_name}' has {count} documents")
                
                # Show sample entries
                if count > 0:
                    results = collection.get(limit=2)
                    print(f"\n📋 Sample entries:")
                    for i, doc_id in enumerate(results['ids']):
                        metadata = results['metadatas'][i] if results['metadatas'] else {}
                        print(f"   [{i+1}] ID: {doc_id}")
                        print(f"       Metadata: {metadata}")
                        print(f"       Content: {results['documents'][i][:100]}...")
        
        except Exception as e:
            print(f"❌ Error testing ChromaDB: {e}")


def main():
    """Main function."""
    print("📚 Turbonomic API Documentation to ChromaDB Processor")
    print("=" * 60)
    
    # Initialize processor
    processor = TurbonomicPDFProcessor(
        pdf_path=PDF_PATH,
        chroma_path=CHROMA_PATH,
        collection_name=COLLECTION_NAME
    )
    
    try:
        # Process the PDF
        result = processor.process_pdf()
        
        if result['success']:
            print(f"\n🎉 SUCCESS! Processed PDF and created {result['chunks_created']} chunks")
            
            # Test ChromaDB connection
            processor.test_chromadb_connection()
            
        else:
            print(f"\n❌ FAILED: {result['error']}")
    
    except KeyboardInterrupt:
        print("\n🛑 Processing interrupted by user")
    except Exception as e:
        print(f"\n❌ Unexpected error: {str(e)}")


if __name__ == "__main__":
    main()

📚 Turbonomic API Documentation to ChromaDB Processor
🚀 Starting Turbonomic PDF processing...
📄 Processing 746 pages...


📖 Processed 100/746 pages...
📖 Processed 200/746 pages...
📖 Processed 300/746 pages...
📖 Processed 400/746 pages...
📖 Processed 500/746 pages...
📖 Processed 600/746 pages...
📖 Processed 700/746 pages...
✅ Successfully extracted text from 738 pages
🔄 Creating logical chunks...
💾 Storing 784 chunks in ChromaDB...
✅ Stored batch 1/8
✅ Stored batch 2/8
✅ Stored batch 3/8
✅ Stored batch 4/8
✅ Stored batch 5/8
✅ Stored batch 6/8
✅ Stored batch 7/8
✅ Stored batch 8/8

📊 Processing Statistics:
   📄 Total pages: 746
   ✅ Processed pages: 738
   ⏭️  Skipped pages: 8
   📦 Total chunks: 784
   ❌ Errors: 0

🎉 SUCCESS! Processed PDF and created 784 chunks

🗂️  Collections in ChromaDB:
   - instana_api
   - test_collection
   - concert_api
   - turbo_api

📦 Collection 'turbo_api' has 784 documents

📋 Sample entries:
   [1] ID: page_1_chunk_0
       Metadata: {'chunk_index': 0, 'chunk_type': 'sliding_window', 'page_number': 1, 'source': 'turbonomic_api_docs', 'char_count': 2000}
       Content: } } ],

In [15]:
## TEST THE CREATED CHROMADB 

import chromadb
from chromadb.config import Settings

chroma_path = "/data/wikidata/chromadb" 
collection_name = "turbo_api"
client = chromadb.PersistentClient(path=chroma_path)
collection = client.get_collection(name=collection_name)
collections = client.list_collections()
print("\n🗂 Collections in ChromaDB:")
for col in collections:
    print(f"- {col.name}")

# Load your specific collection (e.g., "turbonomic")
collection = client.get_collection("turbo_api") 

# TO PRINT ALL ENTRIES
print(f"\n📦 Collection '{collection.name}' has {collection.count()} documents.\n")
results = collection.get(limit=3)
print(results)

# TO DELETE A SPECIFIC ENTRY
# bad_id = "032e0fd1d8031a86ec7298652bc62e6b"
# # Delete the entry
# collection.delete(ids=[bad_id])
# print(f"✅ Deleted entry with ID: {bad_id}")

# TO DELETE THE COLLECTION
#client.delete_collection(name="instana_api")


🗂 Collections in ChromaDB:
- instana_api
- test_collection
- concert_api
- turbo_api

📦 Collection 'turbo_api' has 784 documents.

{'ids': ['page_1_chunk_0', 'page_2_chunk_0', 'page_2_chunk_1'], 'embeddings': None, 'documents': ['} } ], "reserveDateTime": "2017-05-04T18:22:12+00:00" } Async placement calculation When you POST a reservation, Turbonomic runs a plan to calculate the optimal placement of the workloads. Depending on the size of the reservation, this can take a significant amount of time. The API calculates the placement asynchronously. If your script relies on the reservation, you should verify that the reservation has completed before moving on. When you execute a reservation, the response to your POST shows that the calculation is in progress: { "uuid": "_0dPrYTDlEeePgeXuo0RRbw", "displayName": "MyReservation", "count": 4, "status": "IN_PROGRESS", "reserveDateTime": "Thu May 04 18:22:12 UTC 2017", "expireDateTime": "Tue Oct 10 12:38:17 UTC 2017" } To examine the placemen

In [14]:
# TEST THE RETRIEVAL

results = collection.query(
    query_texts=["Actions endpoint"],
    n_results=3,
    include=["documents", "metadatas"]
)

for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
    print("\n🔎 Matched Doc:\n", doc)
    print("Metadata:", meta)



🔎 Matched Doc:
 externalActionUrl   type: string   description: URL to third-party software showing this action. Suitable for external action approval executionStatus   description: Status details of the execution of the Action. Include percentage progress, error messages and execution times.   reference: ActionExecutionAuditApiDTO executionCharacteristics   description: Characteristics of the execution of the Action. Include disruptiveness and reversibility.   reference: ActionExecutionCharacteristicApiDTO entityCharacteristics   description: Characteristics of the entity the action is for.   reference: DBEntityAspectApiDTO source   type: string   description: Defines what generated this action.   enum: ['MARKET', 'PARKING', 'BUY_RI', 'CLOUD_OPTIMIZER'] relatedActionsCountByType   type: object   description: [BETA]: Map of related actions count by relationship type that have some relation to this Action   additionalProperties: {'type': 'integer', 'format': 'int32'} actionID   type: i